# Import libraries & define paths

In [33]:
from pathlib import Path
import pandas as pd

In [34]:
DATA_DIR = Path("../data/raw/ESCO_dataset_v1.2.1")

# Load data

In [35]:
tables = {}

for file in DATA_DIR.glob("*.csv"):
    table_name = file.stem
    tables[table_name] = pd.read_csv(file, dtype=str)

tables.keys()

dict_keys(['broaderRelationsOccPillar_en', 'broaderRelationsSkillPillar_en', 'conceptSchemes_en', 'dictionary_en', 'digCompSkillsCollection_en', 'digitalSkillsCollection_en', 'greenShareOcc_en', 'greenSkillsCollection_en', 'ISCOGroups_en', 'languageSkillsCollection_en', 'occupationSkillRelations_en', 'occupations_en', 'researchOccupationsCollection_en', 'researchSkillsCollection_en', 'skillGroups_en', 'skillsHierarchy_en', 'skillSkillRelations_en', 'skills_en', 'transversalSkillsCollection_en'])

# Make df combining occupation <> skill

In [36]:
skills_df = tables["skills_en"]
occupations_df = tables["occupations_en"]
relations_df = tables["occupationSkillRelations_en"]

In [37]:
skills_df.head()

,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage music staff\ncoordinate duties of music...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,manage prison procedures\nmonitor correctional...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,make use of anti-oppressive practices\nuse ant...,NaN,released,2023-11-28T10:45:53.54Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,checking compliance with rolling stock regulat...,NaN,released,2023-11-30T16:29:18.273Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,determine available services\nclassify availab...,NaN,released,2023-11-28T10:38:49.206Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Identify the different services available for ...


In [38]:
esco_all_df = pd.concat(
    tables.values(),
    ignore_index=True,
    sort=False
)

esco_all_df.head()

,conceptType,conceptUri,conceptLabel,broaderType,broaderUri,broaderLabel,conceptSchemeUri,preferredLabel,title,status,...,Description,Scope note,Level 0 code,Level 1 code,Level 2 code,Level 3 code,originalSkillUri,originalSkillType,relatedSkillType,relatedSkillUri
0,ISCOGroup,http://data.europa.eu/esco/isco/C01,Commissioned armed forces officers,ISCOGroup,http://data.europa.eu/esco/isco/C0,Armed forces occupations,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ISCOGroup,http://data.europa.eu/esco/isco/C011,Commissioned armed forces officers,ISCOGroup,http://data.europa.eu/esco/isco/C01,Commissioned armed forces officers,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ISCOGroup,http://data.europa.eu/esco/isco/C0110,Commissioned armed forces officers,ISCOGroup,http://data.europa.eu/esco/isco/C011,Commissioned armed forces officers,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ISCOGroup,http://data.europa.eu/esco/isco/C02,Non-commissioned armed forces officers,ISCOGroup,http://data.europa.eu/esco/isco/C0,Armed forces occupations,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ISCOGroup,http://data.europa.eu/esco/isco/C021,Non-commissioned armed forces officers,ISCOGroup,http://data.europa.eu/esco/isco/C02,Non-commissioned armed forces officers,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
occupations = tables["occupations_en"]
skills = tables["skills_en"]
relations = tables["occupationSkillRelations_en"]

In [40]:
job_skill_df = (
    relations
    .merge(
        occupations[["conceptUri", "preferredLabel", "description"]],
        left_on="occupationUri",
        right_on="conceptUri",
        how="left"
    )
    .merge(
        skills[["conceptUri", "preferredLabel", "description", "skillType", "reuseLevel"]],
        left_on="skillUri",
        right_on="conceptUri",
        how="left",
        suffixes=("_occupation", "_skill")
    )
)

In [41]:
job_skill_df.head()

,occupationUri,occupationLabel,relationType,skillType_occupation,skillUri,skillLabel,conceptUri_occupation,preferredLabel_occupation,description_occupation,conceptUri_skill,preferredLabel_skill,description_skill,skillType_skill,reuseLevel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques,The techniques that facilitate a successful pr...,knowledge,sector-specific
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals,"Manage, schedule and run rehearsals for the pe...",skill/competence,sector-specific
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...,"Assess risks, propose improvements and describ...",skill/competence,sector-specific
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments,Coordinate activities with other artistic and ...,skill/competence,sector-specific
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands,"Work with artists, striving to understand the ...",skill/competence,sector-specific


In [42]:
job_skill_df.shape

(126407, 14)

In [43]:
job_skill_df[[
    "occupationLabel",
    "skillLabel",
    "relationType",
    "skillType_skill",
    "reuseLevel"
]].head(20)

,occupationLabel,skillLabel,relationType,skillType_skill,reuseLevel
0,technical director,theatre techniques,essential,knowledge,sector-specific
1,technical director,organise rehearsals,essential,skill/competence,sector-specific
2,technical director,write risk assessment on performing arts produ...,essential,skill/competence,sector-specific
3,technical director,coordinate with creative departments,essential,skill/competence,sector-specific
4,technical director,adapt to artists' creative demands,essential,skill/competence,sector-specific
5,technical director,negotiate health and safety issues with third ...,essential,skill/competence,cross-sector
6,technical director,adapt designers’ work to the performance venue,essential,skill/competence,sector-specific
7,technical director,promote health and safety,essential,skill/competence,sector-specific
8,technical director,coordinate technical teams in artistic product...,essential,skill/competence,sector-specific
9,technical director,write technical riders,optional,skill/competence,sector-specific


# Quick check for tech skillLabels

In [44]:
ai_pattern = r"\b(artificial intelligence|machine learning|deep learning|neural network|neural networks|computer vision|natural language processing|nlp|data science|data mining|big data)\b"

In [45]:
ai_skills_df = job_skill_df[
    job_skill_df["skillLabel"].str.contains(
        ai_pattern,
        case=False,
        na=False,
        regex=True
    )
].copy()

C:\Users\Lu\AppData\Local\Temp\ipykernel_53968\3261712375.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  job_skill_df["skillLabel"].str.contains(


In [46]:
ai_skills_df[[
    "occupationLabel",
    "skillLabel",
    "relationType",
    "skillType_skill",
    "reuseLevel"
]].head(50)

,occupationLabel,skillLabel,relationType,skillType_skill,reuseLevel
2631,marine engineering technician,data mining,optional,knowledge,sector-specific
2666,marine engineering technician,perform data mining,optional,skill/competence,sector-specific
2670,marine engineering technician,analyse big data,optional,skill/competence,cross-sector
2679,marine engineering technician,utilise machine learning,optional,skill/competence,sector-specific
6569,digital forensics expert,perform data mining,optional,skill/competence,sector-specific
7938,renewable energy engineer,data mining,optional,knowledge,sector-specific
7962,renewable energy engineer,perform data mining,optional,skill/competence,sector-specific
7963,renewable energy engineer,analyse big data,optional,skill/competence,cross-sector
7972,renewable energy engineer,utilise machine learning,optional,skill/competence,sector-specific
10748,customer service representative,data mining methods,optional,knowledge,cross-sector
